# 4.3 複数オブジェクト・合成・責任分担

窓口は対象を探し、機材は自分の状態を守ります。

In [ ]:
def required(value, label):
    cleaned = value.strip()
    if not cleaned:
        raise ValueError(f"{label} must not be empty")
    return cleaned

class EquipmentItem:
    def __init__(self, item_id, name, category, borrower_id=""):
        self.item_id = required(item_id, "item_id")
        self.name = required(name, "name")
        self.category = required(category, "category")
        self.borrower_id = borrower_id.strip() or None

    def is_available(self):
        return self.borrower_id is None

    def loan_to(self, borrower_id):
        borrower_id = required(borrower_id, "borrower_id")
        if not self.is_available():
            raise ValueError("item is already on loan")
        self.borrower_id = borrower_id

    def return_item(self):
        if self.is_available():
            raise ValueError("item is not on loan")
        self.borrower_id = None

    def to_record(self):
        return {"item_id": self.item_id, "name": self.name,
                "category": self.category, "borrower_id": self.borrower_id or ""}

class LendingDesk:
    def __init__(self):
        self.items = {}

    def add_item(self, item):
        if not isinstance(item, EquipmentItem):
            raise TypeError("item must be EquipmentItem")
        if item.item_id in self.items:
            raise ValueError("duplicate item ID")
        self.items[item.item_id] = item

    def find_item(self, item_id):
        return self.items.get(item_id.strip())

    def required_item(self, item_id):
        item = self.find_item(item_id)
        if item is None:
            raise KeyError(item_id)
        return item

    def loan_item(self, item_id, borrower_id):
        self.required_item(item_id).loan_to(borrower_id)

    def return_item(self, item_id):
        self.required_item(item_id).return_item()

    def summary(self):
        available = sum(x.is_available() for x in self.items.values())
        return {"total_items": len(self.items), "available_items": available,
                "loaned_items": len(self.items) - available}

## コレクションへ追加して検索する

In [ ]:
counter = LendingDesk()
counter.add_item(EquipmentItem("E001", "Laptop", "Computer"))
counter.add_item(EquipmentItem("E002", "Projector", "Presentation", "M021"))
print(counter.find_item(" E001 ").to_record())
print(counter.find_item("E999"))

## 委譲を追跡する

窓口経由の貸出では、窓口が検索し、機材が遷移を検証します。

In [ ]:
counter.loan_item("E001", "M014")
try:
    counter.loan_item("E002", "M022")
except ValueError as error:
    print("REJECTED:", error)
print(counter.summary())

## エラーの意味を分ける

In [ ]:
for item_id in ["E002", "E999"]:
    try:
        counter.loan_item(item_id, "M100")
    except KeyError:
        print(item_id, "UNKNOWN")
    except ValueError:
        print(item_id, "INVALID STATE")

## 統合練習

E003を追加し、貸出、返却、集計を行います。各行で検索・一意性・状態遷移の責任がどこにあるか確認してください。